# Can UrbanM2M reproduce Suzhou urban expansion from 2010 to 2013?

This case frames UrbanM2M as a small validation experiment. The research question is whether a live OpenGMS UrbanM2M task, driven by the bundled Suzhou inputs, can reproduce the observed 2013 urban pattern from the 2010 landscape and spatial drivers.

The notebook does not read a pre-bundled simulation result as the evidence path. It submits a fresh task, downloads the returned simulation and probability rasters, and evaluates those live outputs against observed 2013 urban land.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import zipfile

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import ListedColormap
from pygeomodel import GeoModeler


def configure_opengms_http_timeout(default_seconds=180):
    """Let slow OpenGMS manager endpoints respond before the SDK gives up."""
    timeout_seconds = int(os.getenv("OPENGMS_LIVE_TIMEOUT_SECONDS", str(default_seconds)) or default_seconds)
    try:
        from ogmsServer2.openUtils.http_client import HttpClient
    except Exception as exc:
        print(f"OpenGMS SDK timeout helper was not applied: {exc}")
        return timeout_seconds

    if getattr(HttpClient, "_opengeolab_timeout_patch", False):
        return timeout_seconds

    original_get_sync = HttpClient.get_sync
    original_get_file_sync = HttpClient.get_file_sync
    original_post_sync = HttpClient.post_sync

    def get_sync(url, timeout=None, params=None, headers=None):
        return original_get_sync(url=url, timeout=timeout or timeout_seconds, params=params, headers=headers)

    def get_file_sync(url, timeout=None, params=None, headers=None):
        return original_get_file_sync(url=url, timeout=timeout or timeout_seconds, params=params, headers=headers)

    def post_sync(url, timeout=None, data=None, json=None, files=None, headers=None):
        return original_post_sync(url=url, timeout=timeout or timeout_seconds, data=data, json=json, files=files, headers=headers)

    HttpClient.get_sync = staticmethod(get_sync)
    HttpClient.get_file_sync = staticmethod(get_file_sync)
    HttpClient.post_sync = staticmethod(post_sync)
    HttpClient._opengeolab_timeout_patch = True
    print(f"OpenGMS SDK HTTP timeout: {timeout_seconds}s")
    return timeout_seconds


DATA_DIR = Path("data")
MODEL_ID = "d65631a5-9e1a-4450-9487-640c5a6494c2"
MODEL_NAME = "UrbanM2M城市扩张模拟模型"

MODEL_INPUT_ZIP = DATA_DIR / "Suzhou.zip"
RANGE_RASTER = DATA_DIR / "Suzhou-Masked" / "range.tif"
RESTRICTION_RASTER = DATA_DIR / "Suzhou-Masked" / "restriction.tif"
SLOPE_RASTER = DATA_DIR / "Suzhou-Masked" / "vars0" / "slope.tif"
TOWN_DISTANCE_RASTER = DATA_DIR / "Suzhou-Masked" / "vars0" / "town.tif"
COUNTY_DISTANCE_RASTER = DATA_DIR / "Suzhou-Masked" / "vars0" / "county.tif"
LAND_2010_RASTER = DATA_DIR / "Suzhou-Masked" / "year" / "land_2010.tif"
LAND_2013_RASTER = DATA_DIR / "Suzhou-Masked" / "year" / "land_2013.tif"
BOUNDARY_SHP = DATA_DIR / "Suzhou" / "sz.shp"
REGION_SHP = DATA_DIR / "Around-Taihu-Lake" / "yrdbuffer.shp"

LIVE_OUTPUT_DIR = Path("outputs/live-urban-m2m")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = LIVE_OUTPUT_DIR / RUN_ID
DOWNLOAD_DIR = RUN_DIR / "downloads"
EXTRACT_DIR = RUN_DIR / "extracted"

required_inputs = [
    MODEL_INPUT_ZIP,
    RANGE_RASTER,
    RESTRICTION_RASTER,
    SLOPE_RASTER,
    TOWN_DISTANCE_RASTER,
    COUNTY_DISTANCE_RASTER,
    LAND_2010_RASTER,
    LAND_2013_RASTER,
    BOUNDARY_SHP,
    REGION_SHP,
]
for required_path in required_inputs:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required case input: {required_path}")

RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Live task artifacts for this run will be stored in: {RUN_DIR}")

## 1. Study Area and Observed Change

Suzhou is used as a compact validation setting because both the historical land rasters and the spatial driving variables are available locally. We first inspect the study area and estimate the observed urban growth between 2010 and 2013 before asking UrbanM2M to simulate the same period.

In [ ]:
suzhou = gpd.read_file(BOUNDARY_SHP)
region = gpd.read_file(REGION_SHP)

fig, ax = plt.subplots(figsize=(6, 6))
region.to_crs(suzhou.crs).boundary.plot(ax=ax, color="#94a3b8", linewidth=1.2, label="Taihu context")
suzhou.boundary.plot(ax=ax, color="#0f172a", linewidth=1.5, label="Suzhou")
ax.set_title("Suzhou study area")
ax.set_axis_off()
ax.legend(loc="lower right")

In [ ]:
def read_masked_raster(path):
    with rasterio.open(path) as src:
        array = src.read(1, masked=True)
        profile = src.profile.copy()
        bounds = src.bounds
        extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
        transform = src.transform
    return array, profile, extent, transform

land_2010, land_2010_profile, land_extent, land_transform = read_masked_raster(LAND_2010_RASTER)
land_2013, land_2013_profile, _, _ = read_masked_raster(LAND_2013_RASTER)
slope, _, _, _ = read_masked_raster(SLOPE_RASTER)

land_cmap = ListedColormap(["#f8fafc", "#1e293b"])
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(land_2010, cmap=land_cmap, interpolation="nearest")
axes[0].set_title("Observed urban land, 2010")
axes[1].imshow(land_2013, cmap=land_cmap, interpolation="nearest")
axes[1].set_title("Observed urban land, 2013")
im = axes[2].imshow(slope, cmap="terrain", interpolation="nearest")
axes[2].set_title("Slope driver")
for ax in axes:
    ax.set_axis_off()
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()

urban_2010_pixels = int((land_2010.compressed() == 1).sum())
urban_2013_pixels = int((land_2013.compressed() == 1).sum())
observed_growth_pixels = urban_2013_pixels - urban_2010_pixels
observed_summary = {
    "urban_pixels_2010": urban_2010_pixels,
    "urban_pixels_2013": urban_2013_pixels,
    "observed_growth_pixels": observed_growth_pixels,
}
display(pd.DataFrame([observed_summary]))

## 2. Configure the UrbanM2M Experiment

The live model task uses the local Suzhou package and explicit raster inputs. The model identifier is kept visible so the experiment can be traced back to the OpenGMS model item.

In [ ]:
modeler = GeoModeler()
model = modeler.get_model(MODEL_NAME)
if model.model_id and model.model_id != MODEL_ID:
    raise RuntimeError(f"Unexpected UrbanM2M model id: {model.model_id}")

params = {
    "research_range": str(RANGE_RASTER),
    "research_restriction": str(RESTRICTION_RASTER),
    "history_land_data": str(MODEL_INPUT_ZIP),
    "slope_data": str(SLOPE_RASTER),
    "dist_town_data": str(TOWN_DISTANCE_RASTER),
    "dist_county_data": str(COUNTY_DISTANCE_RASTER),
    "st_year": 2010,
    "first_sim_year": 2013,
    "out_len": 1,
    "land_demands": "1000",
}

model_inputs = pd.DataFrame([item.to_dict() for item in model.inputs])
print(model.display_name or model.name)
print(model.model_id)
display(model_inputs[["name", "event_name", "data_type", "is_file", "required"]])
params

## 3. Submit the Live OpenGMS UrbanM2M Task

This is the model execution cell. It waits for the OpenGMS service response, downloads the returned files into the run directory, and writes a JSON task record. No cached simulation raster is substituted if the service fails or returns no files.

In [ ]:
configure_opengms_http_timeout()

print("Submitting live OpenGMS UrbanM2M task...")
live_result = modeler.invoke(
    MODEL_NAME,
    params=params,
    output_dir=DOWNLOAD_DIR,
    record_path=RUN_DIR / "task_result.json",
)

if not live_result.downloaded_outputs:
    raise RuntimeError("The OpenGMS task completed without downloadable outputs; no historical output will be substituted.")

print(f"Task status: {live_result.status}")
print(f"Downloaded {len(live_result.downloaded_outputs)} output file(s).")
display(pd.DataFrame(live_result.outputs))
live_result.downloaded_outputs

## 4. Resolve Live Simulation and Probability Rasters

UrbanM2M returns a simulated urban-land raster package and a transition-probability package. The helper below unpacks the current task outputs and finds the 2013 simulation and probability rasters from those returned files.

In [ ]:
def unpack_downloaded_outputs(downloaded_outputs, extract_dir):
    extract_dir.mkdir(parents=True, exist_ok=True)
    discovered_files = []
    for item in downloaded_outputs:
        downloaded_path = Path(item)
        if not downloaded_path.exists():
            raise FileNotFoundError(f"Downloaded output was not found: {downloaded_path}")
        if downloaded_path.suffix.lower() == ".zip":
            target_dir = extract_dir / downloaded_path.stem
            target_dir.mkdir(parents=True, exist_ok=True)
            root = target_dir.resolve()
            with zipfile.ZipFile(downloaded_path) as archive:
                for member in archive.infolist():
                    member_target = (target_dir / member.filename).resolve()
                    if member_target != root and root not in member_target.parents:
                        raise RuntimeError(f"Blocked unexpected zip member path: {member.filename}")
                    archive.extract(member, target_dir)
            discovered_files.extend([path for path in target_dir.rglob("*") if path.is_file()])
        else:
            discovered_files.append(downloaded_path)
    return sorted(discovered_files)


def find_live_raster(paths, preferred_terms):
    tifs = [path for path in paths if path.suffix.lower() in {".tif", ".tiff"}]
    for term in preferred_terms:
        preferred = [path for path in tifs if term.lower() in path.name.lower()]
        if preferred:
            return sorted(preferred)[0]
    if tifs:
        return sorted(tifs)[0]
    available = ", ".join(str(path) for path in paths[:20])
    raise FileNotFoundError(f"No live GeoTIFF output was found. Available files: {available}")

live_files = unpack_downloaded_outputs(live_result.downloaded_outputs, EXTRACT_DIR)
sim_2013_path = find_live_raster(live_files, preferred_terms=("sim_2013", "sim"))
prob_2013_path = find_live_raster(live_files, preferred_terms=("prob_2013", "prob"))

print(f"Using live simulation raster: {sim_2013_path}")
print(f"Using live probability raster: {prob_2013_path}")

## 5. Evaluate the Live Model Output

The simulated 2013 raster is compared with the observed 2013 raster over their shared valid mask. Precision, recall, and F1-score are therefore properties of the live model output from this task.

In [ ]:
sim_2013, sim_profile, _, sim_transform = read_masked_raster(sim_2013_path)
prob_2013, prob_profile, _, _ = read_masked_raster(prob_2013_path)

if sim_2013.shape != land_2013.shape:
    raise RuntimeError(f"Simulation shape {sim_2013.shape} does not match observed 2013 shape {land_2013.shape}")
if prob_2013.shape != land_2013.shape:
    raise RuntimeError(f"Probability shape {prob_2013.shape} does not match observed 2013 shape {land_2013.shape}")

base = np.ma.filled(land_2010, 0)
simulated = np.ma.filled(sim_2013, 0)
observed = np.ma.filled(land_2013, 0)
valid_mask = (~np.ma.getmaskarray(land_2013)) & (~np.ma.getmaskarray(sim_2013))

truth = observed[valid_mask]
prediction = simulated[valid_mask]
true_positive = int(((truth == 1) & (prediction == 1)).sum())
false_negative = int(((truth == 1) & (prediction == 0)).sum())
false_positive = int(((truth == 0) & (prediction == 1)).sum())
true_negative = int(((truth == 0) & (prediction == 0)).sum())

precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else 0.0
recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
accuracy = (true_positive + true_negative) / len(truth) if len(truth) else 0.0

pixel_area_km2 = abs(sim_transform.a * sim_transform.e) / 1_000_000
sim_urban_pixels = int((sim_2013.compressed() == 1).sum())
new_expansion_pixels = int(((base == 0) & (simulated == 1) & valid_mask).sum())

validation_summary = {
    "run_id": RUN_ID,
    "simulation_raster": str(sim_2013_path),
    "probability_raster": str(prob_2013_path),
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "true_positive_pixels": true_positive,
    "false_negative_pixels": false_negative,
    "false_positive_pixels": false_positive,
    "simulated_urban_area_km2": sim_urban_pixels * pixel_area_km2,
    "new_simulated_expansion_area_km2": new_expansion_pixels * pixel_area_km2,
}
display(pd.DataFrame([validation_summary]).T.rename(columns={0: "value"}))

In [ ]:
change_map = np.zeros_like(base)
change_map[base == 1] = 1
change_map[(base == 0) & (simulated == 1)] = 2

validation_map = np.zeros_like(base)
validation_map[(observed == 1) & (simulated == 1)] = 1
validation_map[(observed == 1) & (simulated == 0)] = 2
validation_map[(observed == 0) & (simulated == 1)] = 3
validation_map[~valid_mask] = 0

change_cmap = ListedColormap(["#f8fafc", "#26384d", "#ef4444"])
validation_cmap = ListedColormap(["#ffffff", "#2563eb", "#facc15", "#dc2626"])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(change_map, cmap=change_cmap, interpolation="nearest")
axes[0].set_title("Live simulated expansion")
prob_plot = axes[1].imshow(prob_2013, cmap="magma", interpolation="nearest")
axes[1].set_title("Live transition probability")
axes[2].imshow(validation_map, cmap=validation_cmap, interpolation="nearest")
axes[2].set_title("Observed vs. simulated 2013")

for ax in axes:
    ax.set_axis_off()
axes[0].legend(
    handles=[
        mpatches.Patch(color="#26384d", label="Urban in 2010"),
        mpatches.Patch(color="#ef4444", label="New simulated expansion"),
    ],
    loc="lower right",
)
axes[2].legend(
    handles=[
        mpatches.Patch(color="#2563eb", label="Hit"),
        mpatches.Patch(color="#facc15", label="Miss"),
        mpatches.Patch(color="#dc2626", label="False alarm"),
    ],
    loc="lower right",
)
fig.colorbar(prob_plot, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()

In [ ]:
summary_path = RUN_DIR / "validation_summary.json"
summary_path.write_text(json.dumps(validation_summary, indent=2), encoding="utf-8")
print(f"Saved validation summary: {summary_path}")

## 6. Interpretation

The validation statistics are tied to the current OpenGMS task record and downloaded rasters in the run directory. A later run can be compared by its saved task record and validation summary. This keeps the case close to a real modeling paper workflow: define a question, submit the model, inspect the returned result, validate against observations, and state the uncertainty from the live run.